# CNN para clasificar el radical de kanji manuscritos (ETL9B)

Este notebook reúne **todo el proyecto `kanji_cnn`** en un único documento ejecutable de principio a fin:

1. Configuración
2. El formato ETL9B
3. Los 214 radicales Kangxi
4. Etiquetado kanji → radical con IDS
5. Construcción del dataset balanceado
6. Dataset de PyTorch y augmentación
7. División train / val / test por escritor
8. Arquitectura de la CNN
9. Entrenamiento
10. Curvas de aprendizaje
11. Evaluación en test
12. Comparación con ResNet50
13. Inferencia
14. Conclusiones

**Objetivo.** Dada la imagen de un kanji escrito a mano, predecir su *radical visual*: el componente gráfico que lo clasifica según el sistema Kangxi (por ejemplo, 海 → 氵/水, 語 → 言).

### Cómo ejecutarlo

- Coloca los archivos `ETL9B_1` … `ETL9B_5` en la carpeta `dataset/`, junto a este notebook (`src/kanji_cnn/notebook/dataset/`). También se aceptan dentro de `dataset/ETL9B/`. ETL9B no se redistribuye: se solicita en la [ETL Character Database](http://etlcdb.db.aist.go.jp/) (AIST).
- Abre el notebook con su carpeta como directorio de trabajo (es lo que hacen Jupyter y VS Code por defecto).
- Dependencias: `torch`, `torchvision`, `numpy`, `pillow`, `scikit-learn` y `matplotlib` (las del `pyproject.toml`), más un kernel de Jupyter (`ipykernel`).
- La primera ejecución descarga `ids.txt` de [cjkvi-ids](https://github.com/cjkvi/cjkvi-ids), así que necesita internet.
- Lo que se genera (índice del dataset, modelos, historiales, métricas) se guarda en `artifacts/`. Si ya existe un modelo entrenado, se carga en vez de reentrenar (ver `RETRAIN` y `RETRAIN_RESNET`).
- La sección 12 descarga los pesos de ResNet50 preentrenada en ImageNet (~100 MB) solo si tiene que entrenarla.

## 1. Configuración

Imports, rutas e hiperparámetros. Todo lo ajustable está en esta celda.

In [ ]:
from __future__ import annotations

import copy
import json
import random
import re
import struct
import time
import urllib.request
from collections import defaultdict
from dataclasses import dataclass
from itertools import islice
from pathlib import Path
from typing import Callable, Iterable, Iterator, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from matplotlib import font_manager
from matplotlib.ticker import MaxNLocator
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import ResNet50_Weights, resnet50

In [ ]:
# --- Rutas (relativas a la carpeta del notebook) ---
NOTEBOOK_DIR = Path.cwd()
DATASET_DIR = NOTEBOOK_DIR / "dataset"
ETL_FILENAMES = [f"ETL9B_{i}" for i in range(1, 6)]
# Los archivos ETL9B pueden estar directamente en dataset/ o en dataset/ETL9B/
ETL_ROOT = DATASET_DIR / "ETL9B" if (DATASET_DIR / "ETL9B" / "ETL9B_1").exists() else DATASET_DIR
ETL_PATHS = [ETL_ROOT / name for name in ETL_FILENAMES]

ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"   # todo lo generado por el notebook
BUILD_DIR = ARTIFACT_DIR / "build"           # índice del dataset y etiquetas
RUN_DIR = ARTIFACT_DIR / "run"               # modelo, historial y métricas de la CNN
RESNET_DIR = ARTIFACT_DIR / "resnet50"       # lo mismo para la ResNet50 (sección 12)
IDS_CACHE_PATH = ARTIFACT_DIR / "ids.txt"

# --- Construcción del dataset ---
BUILD_SEED = 1234              # muestreo reproducible al balancear clases
MIN_SAMPLES_PER_CLASS = 500    # un radical necesita al menos estas muestras para ser clase
MAX_SAMPLES_PER_CLASS = 1000   # y se submuestrea hasta este tope
MAX_CLASSES = 50               # se conservan los 50 radicales con más muestras

# --- Entrenamiento ---
SPLIT_SEED = 42      # split train/val/test y semilla de PyTorch
EPOCHS = 20          # en CPU cada época tarda ~7-8 min
RETRAIN = False      # True: reentrena aunque exista artifacts/run/model.pt
LR_CNN = 1e-3

# --- Comparación con ResNet50 (sección 12) ---
EPOCHS_RESNET = EPOCHS   # mismas épocas que la CNN; en CPU cada una tarda ~14 min
LR_RESNET = 1e-4         # fine-tuning: un lr alto destruiría los pesos de ImageNet
RETRAIN_RESNET = False   # True: reentrena aunque exista artifacts/resnet50/model.pt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

missing = [p for p in ETL_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan archivos de ETL9B: " + ", ".join(str(p) for p in missing)
        + "\nColócalos en la carpeta dataset/ junto al notebook."
    )
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"ETL9B en: {ETL_ROOT}")
print(f"Dispositivo: {DEVICE}")

In [ ]:
# Fuente CJK para que matplotlib pueda dibujar los kanji en títulos y ejes.
# Sin ella, las etiquetas muestran solo el número de radical (sin glifo).
CJK_FONT = "Noto Sans CJK JP"
try:
    font_manager.findfont(CJK_FONT, fallback_to_default=False)
    plt.rcParams["font.family"] = [CJK_FONT, "DejaVu Sans"]
    HAS_CJK_FONT = True
except ValueError:
    HAS_CJK_FONT = False
    print(f"Aviso: no se encontró la fuente '{CJK_FONT}'; los kanji no se dibujarán en las gráficas.")

## 2. El formato ETL9B

ETL9B es una base de datos de caracteres japoneses manuscritos del AIST: **3036 caracteres** (2965 kanji JIS nivel 1 y 71 hiragana), escritos por **4000 personas**, y repartidos en 5 archivos binarios.

Cada archivo es una secuencia de **registros de 576 bytes**. El primero es una cabecera vacía. El resto tiene esta estructura:

| bytes | contenido |
|---|---|
| 0–1 | número de hoja / id del escritor (big-endian) |
| 2–3 | código JIS X 0208 del carácter |
| 4–7 | lectura en romaji (ASCII) |
| 8–511 | imagen binaria de 64×63 píxeles, 1 bit por píxel (504 bytes) |
| 512–575 | sin uso |

Como los registros tienen tamaño fijo, cualquier muestra se localiza por `(archivo, offset)`. Por eso no hace falta cargar las imágenes en memoria: basta con guardar su posición.

In [ ]:
RECORD_SIZE = 576      # cada registro ocupa 576 bytes
HEADER_SIZE = 576      # el primer registro del archivo es vacío y se salta
IMAGE_SIZE = (64, 63)  # ancho × alto de la imagen
IMAGE_BYTES = 504      # 64*63 bits / 8


def jis_to_char(jis: int) -> str:
    """Convierte un código JIS X 0208 en el carácter Unicode (pasando por EUC-JP)."""
    return bytes([(jis >> 8) | 0x80, (jis & 0xFF) | 0x80]).decode("euc_jp")


@dataclass(frozen=True)
class EtlRecord:
    file_index: int               # en cuál de los 5 archivos está
    byte_offset: int              # posición del registro dentro del archivo
    writer_id: int                # id del escritor
    jis_code: int                 # código JIS del carácter
    char: str                     # carácter imprimible
    tag: str                      # lectura en romaji
    image: Optional[Image.Image]  # imagen (None si no se pidió)


def decode_record(chunk: bytes, file_index: int, byte_offset: int, *, with_image: bool = True) -> EtlRecord:
    """Decodifica un registro de 576 bytes."""
    writer_id, jis = struct.unpack(">HH", chunk[:4])
    image = Image.frombytes("1", IMAGE_SIZE, chunk[8 : 8 + IMAGE_BYTES]) if with_image else None
    return EtlRecord(file_index, byte_offset, writer_id, jis, jis_to_char(jis), chunk[4:8].decode("ascii"), image)


def iter_records(path: Path, file_index: int, *, with_image: bool = True) -> Iterator[EtlRecord]:
    """Recorre todos los registros de un archivo ETL9B."""
    with open(path, "rb") as f:
        f.read(HEADER_SIZE)
        offset = HEADER_SIZE
        while True:
            chunk = f.read(RECORD_SIZE)
            if len(chunk) < RECORD_SIZE:
                break
            yield decode_record(chunk, file_index, offset, with_image=with_image)
            offset += RECORD_SIZE


def iter_all_records(paths: Sequence[Path], **kwargs) -> Iterator[EtlRecord]:
    """Recorre los registros de todos los archivos, en orden."""
    for file_index, path in enumerate(paths):
        yield from iter_records(path, file_index, **kwargs)


def read_record_at(paths: Sequence[Path], file_index: int, byte_offset: int) -> EtlRecord:
    """Lee un único registro a partir de su posición (acceso aleatorio)."""
    with open(paths[file_index], "rb") as f:
        f.seek(byte_offset)
        chunk = f.read(RECORD_SIZE)
    return decode_record(chunk, file_index, byte_offset, with_image=True)


def image_to_array(image: Image.Image) -> np.ndarray:
    """Imagen de 1 bit → matriz uint8 de 63×64 con valores 0/1 (1 = trazo)."""
    return np.array(image, dtype=np.uint8)


for path in ETL_PATHS:
    n = (path.stat().st_size - HEADER_SIZE) // RECORD_SIZE
    print(f"{path.name}: {path.stat().st_size / 2**20:.1f} MiB, {n} registros")

Algunos registros del primer archivo, tomados cada 150 posiciones para ver caracteres variados:

In [ ]:
sample_records = list(islice(iter_records(ETL_PATHS[0], 0), 0, 24 * 150, 150))
fig, axes = plt.subplots(3, 8, figsize=(13, 5.5))
for ax, rec in zip(axes.flat, sample_records):
    ax.imshow(image_to_array(rec.image), cmap="gray_r")
    label = rec.char if HAS_CJK_FONT else rec.tag.strip()
    ax.set_title(f"{label}  (escritor {rec.writer_id})", fontsize=9)
    ax.axis("off")
fig.suptitle("Muestras de ETL9B_1")
plt.tight_layout()
plt.show()

## 3. Los 214 radicales Kangxi

El diccionario Kangxi clasifica los caracteres chinos y japoneses en **214 radicales**. Muchos radicales tienen **variantes gráficas** según su posición en el carácter: 水 aparece como 氵 a la izquierda (海), 人 como 亻 (休), 手 como 扌 (持). Para etiquetar por *apariencia*, todas las variantes de un radical se agrupan bajo un mismo índice.

In [ ]:
@dataclass(frozen=True)
class RadicalEntry:
    index: int
    main_form: str
    variants: tuple[str, ...] = ()

#Diccionario de radicales según la clasificación KANGXI
KANGXI_RADICALS: list[RadicalEntry] = [
    RadicalEntry(1, "一"),
    RadicalEntry(2, "丨"),
    RadicalEntry(3, "丶"),
    RadicalEntry(4, "丿", ("乀", "乁")),
    RadicalEntry(5, "乙", ("乚", "乛")),
    RadicalEntry(6, "亅"),
    RadicalEntry(7, "二"),
    RadicalEntry(8, "亠"),
    RadicalEntry(9, "人", ("亻",)),
    RadicalEntry(10, "儿"),
    RadicalEntry(11, "入"),
    RadicalEntry(12, "八"),
    RadicalEntry(13, "冂"),
    RadicalEntry(14, "冖"),
    RadicalEntry(15, "冫"),
    RadicalEntry(16, "几"),
    RadicalEntry(17, "凵"),
    RadicalEntry(18, "刀", ("刁", "刂")),
    RadicalEntry(19, "力"),
    RadicalEntry(20, "勹"),
    RadicalEntry(21, "匕"),
    RadicalEntry(22, "匚"),
    RadicalEntry(23, "匸"),
    RadicalEntry(24, "十"),
    RadicalEntry(25, "卜"),
    RadicalEntry(26, "卩"),
    RadicalEntry(27, "厂"),
    RadicalEntry(28, "厶"),
    RadicalEntry(29, "又"),
    RadicalEntry(30, "口"),
    RadicalEntry(31, "囗"),
    RadicalEntry(32, "土"),
    RadicalEntry(33, "士"),
    RadicalEntry(34, "夂"),
    RadicalEntry(35, "夊"),
    RadicalEntry(36, "夕"),
    RadicalEntry(37, "大", ("夨",)),
    RadicalEntry(38, "女"),
    RadicalEntry(39, "子", ("孑", "孒", "孓")),
    RadicalEntry(40, "宀"),
    RadicalEntry(41, "寸"),
    RadicalEntry(42, "小"),
    RadicalEntry(43, "尢", ("尣",)),
    RadicalEntry(44, "尸"),
    RadicalEntry(45, "屮"),  # Unihan's first pick 䶹 is an obscure duplicate
    RadicalEntry(46, "山"),
    RadicalEntry(47, "巛", ("川",)),
    RadicalEntry(48, "工"),
    RadicalEntry(49, "己", ("已", "巳")),
    RadicalEntry(50, "巾"),
    RadicalEntry(51, "干"),
    RadicalEntry(52, "幺", ("乡",)),
    RadicalEntry(53, "广"),
    RadicalEntry(54, "廴"),
    RadicalEntry(55, "廾"),
    RadicalEntry(56, "弋"),
    RadicalEntry(57, "弓"),
    RadicalEntry(58, "彐", ("彑",)),
    RadicalEntry(59, "彡"),
    RadicalEntry(60, "彳"),
    RadicalEntry(61, "心", ("忄",)),
    RadicalEntry(62, "戈"),
    RadicalEntry(63, "戶", ("户", "戸")),
    RadicalEntry(64, "手", ("扌", "才", "龵")),
    RadicalEntry(65, "支"),
    RadicalEntry(66, "攴", ("攵",)),
    RadicalEntry(67, "文"),
    RadicalEntry(68, "斗"),
    RadicalEntry(69, "斤"),
    RadicalEntry(70, "方"),
    RadicalEntry(71, "无"),
    RadicalEntry(72, "日"),
    RadicalEntry(73, "曰"),
    RadicalEntry(74, "月"),
    RadicalEntry(75, "木", ("朩",)),
    RadicalEntry(76, "欠"),
    RadicalEntry(77, "止", ("龰",)),
    RadicalEntry(78, "歹"),
    RadicalEntry(79, "殳"),
    RadicalEntry(80, "毋", ("毌",)),
    RadicalEntry(81, "比"),
    RadicalEntry(82, "毛"),
    RadicalEntry(83, "氏"),
    RadicalEntry(84, "气"),
    RadicalEntry(85, "水", ("氵", "氺")),
    RadicalEntry(86, "火", ("灬",)),
    RadicalEntry(87, "爪", ("爫",)),
    RadicalEntry(88, "父"),
    RadicalEntry(89, "爻"),
    RadicalEntry(90, "爿", ("丬",)),
    RadicalEntry(91, "片"),
    RadicalEntry(92, "牙"),  # Unihan's first pick 㸦 is an obscure duplicate
    RadicalEntry(93, "牛", ("牜",)),
    RadicalEntry(94, "犬", ("犭",)),
    RadicalEntry(95, "玄"),
    RadicalEntry(96, "玉", ("王",)),
    RadicalEntry(97, "瓜"),
    RadicalEntry(98, "瓦"),
    RadicalEntry(99, "甘"),
    RadicalEntry(100, "生"),
    RadicalEntry(101, "用", ("甩",)),
    RadicalEntry(102, "田", ("由", "甲", "申", "甴", "电")),
    RadicalEntry(103, "疋"),
    RadicalEntry(104, "疒"),
    RadicalEntry(105, "癶"),
    RadicalEntry(106, "白"),
    RadicalEntry(107, "皮"),
    RadicalEntry(108, "皿"),
    RadicalEntry(109, "目"),
    RadicalEntry(110, "矛"),
    RadicalEntry(111, "矢"),
    RadicalEntry(112, "石"),
    RadicalEntry(113, "示", ("礻",)),
    RadicalEntry(114, "禸"),
    RadicalEntry(115, "禾"),
    RadicalEntry(116, "穴"),
    RadicalEntry(117, "立"),
    RadicalEntry(118, "竹"),
    RadicalEntry(119, "米"),
    RadicalEntry(120, "糸", ("糹",)),
    RadicalEntry(121, "缶"),
    RadicalEntry(122, "网", ("䍏", "罒", "罓")),  # Unihan's first pick 䍏 is obscure
    RadicalEntry(123, "羊"),
    RadicalEntry(124, "羽"),
    RadicalEntry(125, "老", ("考",)),
    RadicalEntry(126, "而"),
    RadicalEntry(127, "耒"),
    RadicalEntry(128, "耳"),
    RadicalEntry(129, "聿"),
    RadicalEntry(130, "肉"),
    RadicalEntry(131, "臣"),
    RadicalEntry(132, "自"),
    RadicalEntry(133, "至"),
    RadicalEntry(134, "臼"),
    RadicalEntry(135, "舌"),
    RadicalEntry(136, "舛"),
    RadicalEntry(137, "舟"),
    RadicalEntry(138, "艮"),
    RadicalEntry(139, "色"),
    RadicalEntry(140, "艸", ("䒑", "艹")),  # Unihan's first pick 䒑 is obscure
    RadicalEntry(141, "虍"),
    RadicalEntry(142, "虫"),
    RadicalEntry(143, "血"),
    RadicalEntry(144, "行"),
    RadicalEntry(145, "衣", ("衤",)),
    RadicalEntry(146, "襾", ("西", "覀")),
    RadicalEntry(147, "見"),
    RadicalEntry(148, "角"),
    RadicalEntry(149, "言", ("訁",)),
    RadicalEntry(150, "谷"),
    RadicalEntry(151, "豆"),
    RadicalEntry(152, "豕"),
    RadicalEntry(153, "豸"),
    RadicalEntry(154, "貝"),
    RadicalEntry(155, "赤"),
    RadicalEntry(156, "走", ("赱",)),
    RadicalEntry(157, "足"),
    RadicalEntry(158, "身"),
    RadicalEntry(159, "車"),
    RadicalEntry(160, "辛"),
    RadicalEntry(161, "辰"),
    RadicalEntry(162, "辵", ("辶",)),
    RadicalEntry(163, "邑"),
    RadicalEntry(164, "酉"),
    RadicalEntry(165, "釆"),
    RadicalEntry(166, "里"),
    RadicalEntry(167, "金", ("釒",)),
    RadicalEntry(168, "長", ("镸",)),
    RadicalEntry(169, "門"),
    RadicalEntry(170, "阜", ("阝",)),
    RadicalEntry(171, "隶"),
    RadicalEntry(172, "隹"),
    RadicalEntry(173, "雨"),
    RadicalEntry(174, "青", ("靑",)),  # 青 is the form actually written in Japan
    RadicalEntry(175, "非"),
    RadicalEntry(176, "面", ("靣",)),
    RadicalEntry(177, "革"),
    RadicalEntry(178, "韋"),
    RadicalEntry(179, "韭"),
    RadicalEntry(180, "音"),
    RadicalEntry(181, "頁"),
    RadicalEntry(182, "風"),
    RadicalEntry(183, "飛"),
    RadicalEntry(184, "食", ("飠",)),
    RadicalEntry(185, "首"),
    RadicalEntry(186, "香"),
    RadicalEntry(187, "馬"),
    RadicalEntry(188, "骨"),
    RadicalEntry(189, "高", ("髙",)),
    RadicalEntry(190, "髟"),
    RadicalEntry(191, "鬥"),
    RadicalEntry(192, "鬯"),
    RadicalEntry(193, "鬲"),
    RadicalEntry(194, "鬼"),
    RadicalEntry(195, "魚"),
    RadicalEntry(196, "鳥"),
    RadicalEntry(197, "鹵"),
    RadicalEntry(198, "鹿"),
    RadicalEntry(199, "麥"),
    RadicalEntry(200, "麻"),
    RadicalEntry(201, "黃", ("黄",)),
    RadicalEntry(202, "黍"),
    RadicalEntry(203, "黑", ("黒",)),
    RadicalEntry(204, "黹"),
    RadicalEntry(205, "黽"),
    RadicalEntry(206, "鼎"),
    RadicalEntry(207, "鼓", ("鼔",)),
    RadicalEntry(208, "鼠"),
    RadicalEntry(209, "鼻"),
    RadicalEntry(210, "齊"),
    RadicalEntry(211, "齒"),
    RadicalEntry(212, "龍"),
    RadicalEntry(213, "龜"),
    RadicalEntry(214, "龠"),
]

assert len(KANGXI_RADICALS) == 214


def build_glyph_to_radical_index() -> dict[str, int]:
    """Devuelve un diccionario de pares (`radical_char,index`) que agrupa variantes de un mismo 
    radical bajo un único índice, usando el sistema de clasificación KANGXI"""
    index: dict[str, int] = {}
    for entry in KANGXI_RADICALS:
        index[entry.main_form] = entry.index
        for glyph in entry.variants:
            index[glyph] = entry.index
    return index


GLYPH_TO_RADICAL = build_glyph_to_radical_index()
RADICAL_GLYPH = {entry.index: entry.main_form for entry in KANGXI_RADICALS}
print(f"{len(KANGXI_RADICALS)} radicales, {len(GLYPH_TO_RADICAL)} glifos (formas principales + variantes)")
print("Ejemplos de variantes:", {g: RADICAL_GLYPH[i] for g, i in GLYPH_TO_RADICAL.items() if g in "氵亻扌忄艹辶阝"})

## 4. Etiquetado kanji → radical con IDS

ETL9B no indica el radical de cada kanji. Para obtenerlo se usan las **IDS** (*Ideographic Description Sequences*) del proyecto [cjkvi-ids](https://github.com/cjkvi/cjkvi-ids), que describen cada carácter como una composición de componentes:

- `語 = ⿰言吾` → 言 a la izquierda, 吾 a la derecha
- `森 = ⿱木林` → 木 arriba y 林 debajo (林 no es un radical, así que 森 se etiqueta como 木)

Los operadores `⿰ ⿱ ⿲ …` (*Ideographic Description Characters*) indican la disposición y el número de componentes, y pueden anidarse. Por eso la cadena se interpreta como un **árbol** de forma recursiva.

**Regla de etiquetado:** se miran solo los componentes del **primer nivel** del árbol. Si **exactamente uno** es un radical Kangxi (o una variante), esa es la etiqueta. Si hay varios (明 = ⿰日月: 日 y 月) o ninguno, el kanji se **excluye** por ambiguo. Así cada muestra tiene una sola clase y el problema es de clasificación multiclase, no multietiqueta.

In [ ]:
IDS_URL = "https://raw.githubusercontent.com/cjkvi/cjkvi-ids/master/ids.txt"

# Etiquetas regionales al final de cada alternativa, p. ej. "⿰言吾[GTJKV]"
_TAG_RE = re.compile(r"(\[[A-Za-z]+\])+$")

# Operadores IDC y el número de componentes que agrupa cada uno
_IDC_ARITY = {"⿰": 2, "⿱": 2, "⿲": 3, "⿳": 3, "⿴": 2, "⿵": 2,
              "⿶": 2, "⿷": 2, "⿸": 2, "⿹": 2, "⿺": 2, "⿻": 2}


class _Node:
    """Nodo del árbol IDS: una hoja (componente) o un operador con hijos."""
    __slots__ = ("char", "children")

    def __init__(self, char: Optional[str], children: list["_Node"]):
        self.char = char
        self.children = children

    @property
    def is_leaf(self) -> bool:
        return not self.children


def _parse_ids(s: str) -> _Node:
    """Convierte una cadena IDS en un árbol (notación prefija, recursiva)."""
    pos = 0

    def parse() -> _Node:
        nonlocal pos
        ch = s[pos]
        pos += 1
        arity = _IDC_ARITY.get(ch)
        if arity is None:
            return _Node(ch, [])
        return _Node(None, [parse() for _ in range(arity)])

    return parse()


def fetch_ids_file(cache_path: Path, *, force: bool = False) -> Path:
    """Descarga ids.txt la primera vez; después usa la copia en caché."""
    if cache_path.exists() and not force:
        return cache_path
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(IDS_URL, timeout=30) as response:
        cache_path.write_bytes(response.read())
    return cache_path


def _pick_alternative(alternatives: list[str]) -> str:
    """ETL9B es japonés: se prefiere la descomposición marcada con J; si no hay, la primera."""
    for alt in alternatives:
        tag_match = _TAG_RE.search(alt)
        if tag_match and "J" in tag_match.group(0):
            return _TAG_RE.sub("", alt)
    return _TAG_RE.sub("", alternatives[0])


def parse_ids_file(path: Path) -> dict[str, str]:
    """Devuelve {carácter: cadena IDS}."""
    ids_map: dict[str, str] = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                ids_map[parts[1]] = _pick_alternative(parts[2:])
    return ids_map


def direct_radical_matches(char: str, ids_map: dict[str, str], glyph_to_radical: dict[str, int]) -> set[int]:
    """Radicales Kangxi entre los componentes de primer nivel de `char`."""
    ids_string = ids_map.get(char)
    if ids_string is None:
        return set()
    try:
        root = _parse_ids(ids_string)
    except IndexError:  # cadena IDS mal formada
        return set()
    slots = [root] if root.is_leaf else root.children
    return {glyph_to_radical[s.char] for s in slots if s.is_leaf and s.char in glyph_to_radical}


def build_kanji_radical_map(kanji_chars: Iterable[str], ids_map: dict[str, str],
                            glyph_to_radical: dict[str, int]) -> dict[str, int]:
    """Asigna a cada kanji su radical, solo si es único (los ambiguos se descartan)."""
    result: dict[str, int] = {}
    for char in kanji_chars:
        matches = direct_radical_matches(char, ids_map, glyph_to_radical)
        if len(matches) == 1:
            result[char] = next(iter(matches))
    return result


IDS_MAP = parse_ids_file(fetch_ids_file(IDS_CACHE_PATH))
print(f"{len(IDS_MAP)} descomposiciones IDS cargadas\n")
for ch in "語海休森明花":
    matches = direct_radical_matches(ch, IDS_MAP, GLYPH_TO_RADICAL)
    verdict = (f"radical {next(iter(matches))} {RADICAL_GLYPH[next(iter(matches))]}"
               if len(matches) == 1 else f"excluido ({len(matches)} radicales)")
    print(f"{ch}  IDS={IDS_MAP.get(ch)}  →  {verdict}")

## 5. Construcción del dataset balanceado

Pasos:

1. **Escanear** los 5 archivos sin decodificar imágenes y agrupar las posiciones de los registros por carácter.
2. **Quitar el hiragana.**
3. **Etiquetar** cada kanji con su radical (sección 4).
4. **Agrupar** las muestras por radical.
5. **Balancear**: se descartan los radicales con menos de `MIN_SAMPLES_PER_CLASS` muestras, se conservan los `MAX_CLASSES` más numerosos y se submuestrea cada uno hasta `MAX_SAMPLES_PER_CLASS`.
6. **Guardar** el índice `sample_index.npy` (una fila por muestra: archivo, offset, escritor, clase) y `class_labels.json` (clase → radical).

Todo el paso tarda unos segundos y, con `BUILD_SEED` fijo, siempre da el mismo resultado.

In [ ]:
SAMPLE_INDEX_PATH = BUILD_DIR / "sample_index.npy"
CLASS_LABELS_PATH = BUILD_DIR / "class_labels.json"
_HIRAGANA_RANGE = ("ぁ", "ゟ")


@dataclass(frozen=True)
class RecordLoc:
    """Posición de un registro: todo lo necesario para volver a leerlo."""
    file_index: int
    byte_offset: int
    writer_id: int


def is_hiragana(ch: str) -> bool:
    return _HIRAGANA_RANGE[0] <= ch <= _HIRAGANA_RANGE[1]


def scan_classes(etl_paths: Sequence[Path]) -> dict[str, list[RecordLoc]]:
    """{carácter: posiciones de todos sus registros}."""
    by_char: dict[str, list[RecordLoc]] = defaultdict(list)
    for record in iter_all_records(etl_paths, with_image=False):
        by_char[record.char].append(RecordLoc(record.file_index, record.byte_offset, record.writer_id))
    return by_char


def filter_kanji(by_char: dict[str, list[RecordLoc]]) -> dict[str, list[RecordLoc]]:
    return {char: locs for char, locs in by_char.items() if not is_hiragana(char)}


def group_by_radical(by_char: dict[str, list[RecordLoc]], kanji_to_radical: dict[str, int]) -> dict[int, list[RecordLoc]]:
    pools: dict[int, list[RecordLoc]] = defaultdict(list)
    for char, locs in by_char.items():
        radical = kanji_to_radical.get(char)
        if radical is not None:
            pools[radical].extend(locs)
    return pools


def balance_classes(pools: dict[int, list[RecordLoc]], *, min_count: int, max_count: int,
                    max_classes: int, seed: int) -> dict[int, list[RecordLoc]]:
    surviving = {radical: locs for radical, locs in pools.items() if len(locs) >= min_count}
    if len(surviving) > max_classes:
        ranked = sorted(surviving.items(), key=lambda kv: len(kv[1]), reverse=True)
        surviving = dict(ranked[:max_classes])
    rng = random.Random(seed)
    balanced: dict[int, list[RecordLoc]] = {}
    for radical, locs in surviving.items():
        ordered = sorted(locs, key=lambda loc: (loc.file_index, loc.byte_offset))
        if len(ordered) > max_count:
            ordered = rng.sample(ordered, max_count)
        balanced[radical] = ordered
    return balanced


def finalize_and_write(balanced: dict[int, list[RecordLoc]], out_dir: Path) -> None:
    """Las clases se numeran 0..N-1 en orden de índice Kangxi."""
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, class_meta = [], []
    for class_index, radical in enumerate(sorted(balanced)):
        locs = balanced[radical]
        class_meta.append({"class_index": class_index, "radical_index": radical,
                           "radical_glyph": RADICAL_GLYPH[radical], "count": len(locs)})
        rows.extend((loc.file_index, loc.byte_offset, loc.writer_id, class_index) for loc in locs)
    np.save(out_dir / "sample_index.npy", np.array(rows, dtype=np.int64))
    (out_dir / "class_labels.json").write_text(json.dumps(class_meta, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
by_char = scan_classes(ETL_PATHS)
print(f"{len(by_char)} caracteres distintos, {sum(len(v) for v in by_char.values())} registros")

kanji_only = filter_kanji(by_char)
print(f"{len(kanji_only)} kanji tras quitar el hiragana")

kanji_to_radical = build_kanji_radical_map(kanji_only.keys(), IDS_MAP, GLYPH_TO_RADICAL)
print(f"{len(kanji_to_radical)} kanji etiquetados, {len(kanji_only) - len(kanji_to_radical)} excluidos (ambiguos / sin radical)")

pools = group_by_radical(kanji_only, kanji_to_radical)
print(f"{len(pools)} radicales con al menos un kanji")

balanced = balance_classes(pools, min_count=MIN_SAMPLES_PER_CLASS, max_count=MAX_SAMPLES_PER_CLASS,
                           max_classes=MAX_CLASSES, seed=BUILD_SEED)
n_floor = sum(1 for locs in pools.values() if len(locs) >= MIN_SAMPLES_PER_CLASS)
print(f"{n_floor} radicales superan las {MIN_SAMPLES_PER_CLASS} muestras; se conservan {len(balanced)}")

finalize_and_write(balanced, BUILD_DIR)
print(f"Dataset: {len(balanced)} clases, {sum(len(v) for v in balanced.values())} muestras → {BUILD_DIR}")

Muestras disponibles por radical antes de balancear. En azul los radicales que se conservan como clase; la línea discontinua marca el mínimo de muestras y la punteada el tope por clase:

In [ ]:
ranked = sorted(pools.items(), key=lambda kv: len(kv[1]), reverse=True)[:80]
names = [f"{r} {RADICAL_GLYPH[r]}" if HAS_CJK_FONT else str(r) for r, _ in ranked]
counts = [len(locs) for _, locs in ranked]
colors = ["tab:blue" if r in balanced else "lightgray" for r, _ in ranked]

fig, ax = plt.subplots(figsize=(16, 4.5))
ax.bar(range(len(ranked)), counts, color=colors)
ax.axhline(MIN_SAMPLES_PER_CLASS, color="tab:red", linestyle="--", label=f"mínimo ({MIN_SAMPLES_PER_CLASS})")
ax.axhline(MAX_SAMPLES_PER_CLASS, color="tab:green", linestyle=":", label=f"tope por clase ({MAX_SAMPLES_PER_CLASS})")
ax.set_xticks(range(len(ranked)), names, rotation=90, fontsize=7)
ax.set_yscale("log")
ax.set(ylabel="muestras (escala log)", title="Muestras por radical (80 radicales más frecuentes)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Dataset de PyTorch y augmentación

`EtlVisualRadicalDataset` solo carga el índice. Cada `__getitem__` lee del disco el registro correspondiente y lo convierte en un tensor `(1, 63, 64)` con valores 0/1.

Para el entrenamiento se aplica una **transformación afín aleatoria** (rotación ±8°, traslación ±8 %, escala 0.9–1.1). Simula las variaciones naturales de la escritura a mano y reduce el sobreajuste.

In [ ]:
def default_transform(img: Image.Image) -> torch.Tensor:
    return torch.from_numpy(image_to_array(img)).unsqueeze(0).float()  # (1, 63, 64)


class EtlVisualRadicalDataset(Dataset):
    def __init__(self, index_path: Path = SAMPLE_INDEX_PATH, class_labels_path: Path = CLASS_LABELS_PATH,
                 etl_paths: Sequence[Path] = ETL_PATHS,
                 transform: Optional[Callable[[Image.Image], torch.Tensor]] = None):
        self._index = np.load(index_path)   # (N, 4): archivo, offset, escritor, clase
        self.classes = self._index[:, 3]
        self.writer_ids = self._index[:, 2]
        self._class_meta = json.loads(Path(class_labels_path).read_text(encoding="utf-8"))
        self._paths = list(etl_paths)
        self.num_classes = len(self._class_meta)
        self.transform = transform or default_transform

    def __len__(self) -> int:
        return len(self._index)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
        file_index, byte_offset, _writer_id, class_index = (int(x) for x in self._index[i])
        record = read_record_at(self._paths, file_index, byte_offset)
        return self.transform(record.image), class_index


AUGMENT = T.RandomAffine(degrees=8, translate=(0.08, 0.08), scale=(0.9, 1.1))

CLASS_LABELS = json.loads(CLASS_LABELS_PATH.read_text(encoding="utf-8"))
CLASS_NAMES = [f"{c['radical_index']} {c['radical_glyph']}" for c in CLASS_LABELS]  # para reportes
PLOT_NAMES = CLASS_NAMES if HAS_CJK_FONT else [str(c["radical_index"]) for c in CLASS_LABELS]

full_dataset = EtlVisualRadicalDataset()
print(f"{len(full_dataset)} muestras, {full_dataset.num_classes} clases, tensor {tuple(full_dataset[0][0].shape)}")

Un ejemplo de cada una de las 50 clases (el título es el radical que se debe predecir):

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(15, 9))
for c, ax in enumerate(axes.flat):
    image, _ = full_dataset[int(np.argmax(full_dataset.classes == c))]
    ax.imshow(image[0], cmap="gray_r")
    ax.set_title(PLOT_NAMES[c], fontsize=10)
    ax.axis("off")
fig.suptitle("Una muestra por clase")
plt.tight_layout()
plt.show()

In [ ]:
original, _ = full_dataset[0]
fig, axes = plt.subplots(1, 9, figsize=(15, 2.2))
axes[0].imshow(original[0], cmap="gray_r")
axes[0].set_title("original")
torch.manual_seed(0)
for ax in axes[1:]:
    ax.imshow(AUGMENT(original)[0], cmap="gray_r")
    ax.set_title("aumentada")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. División train / val / test por escritor

Cada uno de los 4000 escritores escribió muchos caracteres distintos. Si se divide el dataset **por muestra**, la red se evalúa con la letra de personas que ya vio al entrenar y la métrica sale optimista (fuga de información).

Por eso se usa `StratifiedGroupKFold` con 20 *folds* (~5 % cada uno), agrupando por `writer_id`:

- **test** (3 folds, ~15 %): solo se usa al final, para las métricas.
- **val** (3 folds, ~15 %): ajusta la tasa de aprendizaje y elige la mejor época.
- **train** (14 folds, ~70 %): ajuste de pesos.

Los grupos garantizan que **ningún escritor aparezca en dos conjuntos**, y la estratificación mantiene equilibradas las 50 clases en cada uno.

In [ ]:
N_FOLDS, TEST_FOLDS, VAL_FOLDS = 20, 3, 3


def make_splits(dataset: EtlVisualRadicalDataset, seed: int = SPLIT_SEED):
    folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_idx = [fold for _, fold in folds.split(np.zeros(len(dataset)), dataset.classes, dataset.writer_ids)]
    test_idx = np.concatenate(fold_idx[:TEST_FOLDS])
    val_idx = np.concatenate(fold_idx[TEST_FOLDS:TEST_FOLDS + VAL_FOLDS])
    train_idx = np.concatenate(fold_idx[TEST_FOLDS + VAL_FOLDS:])
    return train_idx, val_idx, test_idx


train_idx, val_idx, test_idx = make_splits(full_dataset)
writers = {name: set(full_dataset.writer_ids[idx]) for name, idx in
           (("train", train_idx), ("val", val_idx), ("test", test_idx))}

print(f"{'conjunto':<8} {'muestras':>9} {'%':>6} {'escritores':>11} {'muestras/clase (min-max)':>26}")
for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    per_class = np.bincount(full_dataset.classes[idx], minlength=full_dataset.num_classes)
    print(f"{name:<8} {len(idx):>9} {100 * len(idx) / len(full_dataset):>5.1f}% {len(writers[name]):>11} "
          f"{per_class.min():>19}-{per_class.max()}")

print("\nEscritores compartidos:",
      {"train∩val": len(writers["train"] & writers["val"]),
       "train∩test": len(writers["train"] & writers["test"]),
       "val∩test": len(writers["val"] & writers["test"])})

## 8. Arquitectura de la CNN

`KanjiRadicalCNN` apila **4 bloques convolucionales** que duplican los canales (32 → 64 → 128 → 256) y reducen la resolución a la mitad con *max pooling*. Cada bloque tiene esta forma:

```
[Conv 3×3 → BatchNorm → ReLU] × 2 → MaxPool 2×2
```

Después vienen un **global average pooling**, que hace al modelo independiente de la posición exacta del trazo y reduce mucho los parámetros frente a un `Flatten` directo, y un MLP `256 → 256 → 50` con *Dropout* 0.4.

In [ ]:
def conv_block(feat_in: int, feat_out: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(feat_in, feat_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(feat_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(feat_out, feat_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(feat_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class KanjiRadicalCNN(nn.Module):
    def __init__(self, n_classes: int = 50):
        super().__init__()
        self.feature = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
        )
        self.dense = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.dense(self.feature(x))


demo = KanjiRadicalCNN(len(CLASS_LABELS)).eval()
x = torch.zeros(1, 1, 63, 64)
print(f"entrada            {tuple(x.shape)}")
with torch.no_grad():
    for i, block in enumerate(demo.feature, 1):
        x = block(x)
        print(f"bloque conv {i}      {tuple(x.shape)}")
    print(f"salida (logits)    {tuple(demo.dense(x).shape)}")
print(f"\nParámetros entrenables: {sum(p.numel() for p in demo.parameters() if p.requires_grad):,}")

## 9. Entrenamiento

- **Pérdida**: entropía cruzada con *label smoothing* 0.1, que evita que la red se vuelva sobreconfiada.
- **Optimizador**: Adam con lr = 1e-3 (`LR_CNN`). `ReduceLROnPlateau` divide el lr entre 2 si `val_loss` no mejora en 3 épocas.
- **Mejor época**: se guardan los pesos con menor `val_loss` y se restauran al final, así que el modelo final no es necesariamente el de la última época.
- El historial se escribe en `artifacts/run/history.json` tras cada época. Si se interrumpe, no se pierde.

Si ya existe `artifacts/run/model.pt` y `RETRAIN = False`, se carga ese modelo en lugar de reentrenar.

In [ ]:
def make_loaders(seed: int = SPLIT_SEED):
    """DataLoaders de train (con augmentación), val y test (sin augmentación ni shuffle)."""
    train_full = EtlVisualRadicalDataset(transform=lambda img: AUGMENT(default_transform(img)))
    eval_full = EtlVisualRadicalDataset()
    tr, va, te = make_splits(eval_full, seed)
    return (DataLoader(Subset(train_full, tr), batch_size=64, shuffle=True),
            DataLoader(Subset(eval_full, va), batch_size=128),
            DataLoader(Subset(eval_full, te), batch_size=128))


def make_loss() -> nn.Module:
    return nn.CrossEntropyLoss(label_smoothing=0.1)


def evaluate(model: nn.Module, loader: DataLoader, device: str = DEVICE) -> dict:
    """Loss, accuracy y top-5 sobre `loader`, más las etiquetas reales y predichas."""
    loss_func = make_loss()
    model.eval()
    total_loss, top5 = 0.0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += loss_func(out, yb).item() * len(yb)
            top5 += (out.topk(5, dim=1).indices == yb[:, None]).any(dim=1).sum().item()
            y_true.append(yb.cpu())
            y_pred.append(out.argmax(1).cpu())
    y_true, y_pred = torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()
    n = len(y_true)
    return {"loss": total_loss / n, "acc": float((y_true == y_pred).mean()), "top5": top5 / n,
            "y_true": y_true, "y_pred": y_pred}


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, device: str = DEVICE,
                epochs: int = EPOCHS, lr: float = LR_CNN,
                on_epoch: Optional[Callable[[list[dict]], None]] = None) -> list[dict]:
    model.to(device)
    loss_func = make_loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    history: list[dict] = []
    best_loss, best_state = float("inf"), None

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_func(out, yb)
            loss.backward()
            optimizer.step()
            # métricas de train sobre batches aumentados y con dropout activo
            train_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)

        val = evaluate(model, val_loader, device)
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val["loss"])
        history.append({"epoch": epoch, "train_loss": train_loss / total, "train_acc": correct / total,
                        "val_loss": val["loss"], "val_acc": val["acc"], "lr": lr,
                        "epoch_time": time.perf_counter() - start})
        print(f"época {epoch}/{epochs}: train_loss={train_loss / total:.4f} train_acc={correct / total:.4f} "
              f"val_loss={val['loss']:.4f} val_acc={val['acc']:.4f} lr={lr:.1e}")

        if val["loss"] < best_loss:
            best_loss, best_state = val["loss"], copy.deepcopy(model.state_dict())
        if on_epoch is not None:
            on_epoch(history)

    model.load_state_dict(best_state)
    return history

In [ ]:
CHECKPOINT_PATH = RUN_DIR / "model.pt"
HISTORY_PATH = RUN_DIR / "history.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SPLIT_SEED)
train_loader, val_loader, test_loader = make_loaders(SPLIT_SEED)
model = KanjiRadicalCNN(len(CLASS_LABELS))

if CHECKPOINT_PATH.exists() and not RETRAIN:
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
    best_epoch = ckpt["best_epoch"]
    print(f"Modelo cargado de {CHECKPOINT_PATH} (mejor época {best_epoch}); pon RETRAIN = True para reentrenar.")
else:
    print(f"Entrenando {EPOCHS} épocas en {DEVICE}...")
    history = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR_CNN,
                          on_epoch=lambda h: HISTORY_PATH.write_text(json.dumps(h, indent=2), encoding="utf-8"))
    best_epoch = min(history, key=lambda h: h["val_loss"])["epoch"]
    torch.save({"state_dict": model.state_dict(), "model": "cnn", "n_classes": len(CLASS_LABELS),
                "class_labels": CLASS_LABELS, "seed": SPLIT_SEED, "best_epoch": best_epoch, "lr": LR_CNN},
               CHECKPOINT_PATH)
    print(f"Mejor época: {best_epoch}. Modelo guardado en {CHECKPOINT_PATH}")

model.to(DEVICE)

## 10. Curvas de aprendizaje

Al principio es normal que la accuracy de train quede **por debajo** de la de val: train se mide con imágenes aumentadas y con dropout activo, val no. Una val_loss que sube mientras la de train sigue bajando indica sobreajuste; la línea discontinua marca la época cuyos pesos se conservaron.

In [ ]:
epochs_axis = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, title in ((axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")):
    ax.plot(epochs_axis, [h[f"train_{metric}"] for h in history], marker="o", label="train")
    ax.plot(epochs_axis, [h[f"val_{metric}"] for h in history], marker="o", label="val")
    ax.set(title=title, xlabel="época")
axes[2].plot(epochs_axis, [h["lr"] for h in history], marker="o", color="tab:purple")
axes[2].set(title="Learning rate", xlabel="época", yscale="log")
for ax in axes:
    ax.axvline(best_epoch, color="gray", linestyle="--", label=f"mejor época ({best_epoch})")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Evaluación en test

El conjunto de test (escritores que la red nunca vio) solo se usa aquí. Métricas:

- **accuracy**: proporción de radicales acertados.
- **top-5**: el radical correcto está entre las 5 predicciones más probables.
- **F1 macro**: media del F1 de las 50 clases, todas con el mismo peso.

In [ ]:
test_result = evaluate(model, test_loader)
labels = list(range(len(CLASS_LABELS)))
report = classification_report(test_result["y_true"], test_result["y_pred"], labels=labels,
                               target_names=CLASS_NAMES, output_dict=True, zero_division=0)
metrics = {
    "loss": test_result["loss"],
    "acc": test_result["acc"],
    "top5": test_result["top5"],
    "f1_macro": f1_score(test_result["y_true"], test_result["y_pred"], labels=labels, average="macro", zero_division=0),
    "best_epoch": best_epoch,
    "split_sizes": {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)},
    "report": report,
}
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Test ({len(test_idx)} muestras):  accuracy={metrics['acc']:.4f}  top-5={metrics['top5']:.4f}  "
      f"F1 macro={metrics['f1_macro']:.4f}\n")
print(classification_report(test_result["y_true"], test_result["y_pred"], labels=labels,
                            target_names=CLASS_NAMES, digits=3, zero_division=0))

### Matriz de confusión

Normalizada por fila: la celda (i, j) es la fracción de muestras del radical *i* que se predijeron como *j*. Una diagonal intensa indica un buen modelo; las manchas fuera de ella muestran qué radicales se confunden.

In [ ]:
cm = confusion_matrix(test_result["y_true"], test_result["y_pred"], labels=labels, normalize="true")
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(labels, PLOT_NAMES, rotation=90, fontsize=7)
ax.set_yticks(labels, PLOT_NAMES, fontsize=7)
ax.set(title="Matriz de confusión (test, normalizada por clase real)", xlabel="predicho", ylabel="real")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
print("Confusiones más frecuentes (real → predicho):")
for flat in np.argsort(off_diagonal, axis=None)[::-1][:10]:
    i, j = np.unravel_index(flat, cm.shape)
    print(f"  {CLASS_NAMES[i]:>7} → {CLASS_NAMES[j]:<7} {cm[i, j]:.1%}")

### Accuracy por radical

In [ ]:
recalls = np.array([report[name]["recall"] for name in CLASS_NAMES])
order = np.argsort(recalls)
fig, ax = plt.subplots(figsize=(8, 12))
ax.barh([PLOT_NAMES[i] for i in order], recalls[order], color="tab:blue")
ax.axvline(metrics["acc"], color="tab:red", linestyle="--", label=f"accuracy global ({metrics['acc']:.3f})")
ax.set(title="Accuracy por radical (test)", xlabel="accuracy", xlim=(0, 1))
ax.tick_params(axis="y", labelsize=7)
ax.margins(y=0.01)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Ejemplos mal clasificados

In [ ]:
test_subset = test_loader.dataset   # mismo orden que y_true / y_pred (sin shuffle)
wrong = np.flatnonzero(test_result["y_true"] != test_result["y_pred"])
picked = np.random.default_rng(0).choice(wrong, size=min(32, len(wrong)), replace=False)

fig, axes = plt.subplots(4, 8, figsize=(13, 8))
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, picked):
    image, _ = test_subset[int(i)]
    ax.imshow(image[0], cmap="gray_r")
    ax.set_title(f"real {PLOT_NAMES[test_result['y_true'][i]]}\npred {PLOT_NAMES[test_result['y_pred'][i]]}",
                 fontsize=8, color="tab:red")
fig.suptitle(f"Errores en test ({len(wrong)} de {len(test_subset)})")
plt.tight_layout()
plt.show()

## 12. Comparación con ResNet50

¿Merece la pena una red diseñada para este problema frente a una arquitectura estándar y mucho más grande? Para responderlo, se entrena una **ResNet50 preentrenada en ImageNet** (*transfer learning*) con las mismas condiciones que la CNN y se comparan sobre el mismo test.

**Lo que se mantiene igual:** el split por escritor (semilla 42), las 50 clases, la entrada 1×63×64, la augmentación `RandomAffine` solo en train, la pérdida (CE + *label smoothing* 0.1), Adam con `ReduceLROnPlateau`, el número de épocas y la elección de la mejor época por `val_loss`.

**Única diferencia deliberada: el learning rate.** La CNN parte de pesos aleatorios y usa 1e-3. La ResNet50 parte de pesos ya entrenados y usa 1e-4, el valor habitual en *fine-tuning*: con 1e-3 las primeras actualizaciones destruirían lo aprendido en ImageNet.

### 12.1 Adaptar ResNet50 a kanji de 63×64

ResNet50 espera imágenes RGB de 224×224 normalizadas como ImageNet. Se hacen tres cambios mínimos:

1. **`conv1` de 3 canales a 1.** El filtro nuevo es la **suma** de los tres filtros RGB preentrenados. Sobre una imagen gris replicada en los 3 canales, la convolución original da exactamente lo mismo, así que se conserva lo aprendido.
2. **Normalización** con la media y la desviación del gris de ImageNet (0.449 y 0.226), dentro del propio modelo, para que el pipeline de datos sea idéntico al de la CNN.
3. **Capa final** `fc` de 2048 → 50 radicales, inicializada al azar.

La resolución se deja en 63×64. Reescalar a 128×128 costaría ~38 min por época en CPU y quitar el *stride* del primer bloque, ~150 min.

In [ ]:
class ResNet50Radical(nn.Module):
    """ResNet50 de torchvision para imágenes de 1 canal y `n_classes` radicales."""

    def __init__(self, n_classes: int = 50, pretrained: bool = False):
        super().__init__()
        self.net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        old = self.net.conv1
        self.net.conv1 = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        with torch.no_grad():
            self.net.conv1.weight.copy_(old.weight.sum(dim=1, keepdim=True))  # RGB → gris
        self.net.fc = nn.Linear(self.net.fc.in_features, n_classes)
        self.register_buffer("mean", torch.tensor(0.449))
        self.register_buffer("std", torch.tensor(0.226))

    def forward(self, x):
        return self.net((x - self.mean) / self.std)


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


def size_mb(m: nn.Module) -> float:
    return sum(t.numel() * t.element_size() for t in [*m.parameters(), *m.buffers()]) / 2**20


demo_resnet = ResNet50Radical(len(CLASS_LABELS)).eval()
net = demo_resnet.net
stages = [("conv1 7×7, stride 2", nn.Sequential(net.conv1, net.bn1, net.relu)), ("maxpool 3×3, stride 2", net.maxpool),
          ("layer1 (3 bloques)", net.layer1), ("layer2 (4 bloques)", net.layer2),
          ("layer3 (6 bloques)", net.layer3), ("layer4 (3 bloques)", net.layer4)]
x = torch.zeros(1, 1, 63, 64)
print(f"{'entrada':<24} {tuple(x.shape)}")
with torch.no_grad():
    for name, stage in stages:
        x = stage(x)
        print(f"{name:<24} {tuple(x.shape)}")

print(f"\n{'modelo':<10} {'parámetros':>12} {'tamaño (MB)':>12}")
for name, m in (("CNN", model), ("ResNet50", demo_resnet)):
    print(f"{name:<10} {count_params(m):>12,} {size_mb(m):>12.1f}")

Dos diferencias de diseño saltan a la vista:

- **Resolución.** ResNet50 reduce la imagen de 63×64 a **16×16 en sus dos primeras operaciones** (conv con *stride* 2 + maxpool) y termina en un mapa de **2×2**. La CNN aplica dos convoluciones 3×3 a resolución completa antes de reducir y termina en 3×4. Los trazos de un kanji miden 1–3 píxeles, y la ResNet los resume muy pronto.
- **Tamaño.** ResNet50 tiene unas **19 veces más parámetros**, para solo 35 000 imágenes de entrenamiento.

### 12.2 Entrenar (o cargar) la ResNet50

Si existe `artifacts/resnet50/model.pt` y `RETRAIN_RESNET = False`, se carga. Si no, se entrena con los mismos `train_loader`/`val_loader` de la sección 9; en CPU tarda unos 14 min por época.

In [ ]:
RESNET_CHECKPOINT = RESNET_DIR / "model.pt"
RESNET_HISTORY = RESNET_DIR / "history.json"
RESNET_DIR.mkdir(parents=True, exist_ok=True)

if RESNET_CHECKPOINT.exists() and not RETRAIN_RESNET:
    resnet = ResNet50Radical(len(CLASS_LABELS))   # los pesos vienen del checkpoint, no de ImageNet
    resnet_ckpt = torch.load(RESNET_CHECKPOINT, map_location="cpu")
    resnet.load_state_dict(resnet_ckpt["state_dict"])
    resnet_history = json.loads(RESNET_HISTORY.read_text(encoding="utf-8"))
    resnet_best_epoch = resnet_ckpt["best_epoch"]
    print(f"ResNet50 cargada de {RESNET_CHECKPOINT} (mejor época {resnet_best_epoch}); "
          "pon RETRAIN_RESNET = True para reentrenar.")
else:
    torch.manual_seed(SPLIT_SEED)
    resnet = ResNet50Radical(len(CLASS_LABELS), pretrained=True)
    print(f"Entrenando ResNet50 {EPOCHS_RESNET} épocas en {DEVICE} (lr={LR_RESNET:g})...")
    resnet_history = train_model(resnet, train_loader, val_loader, epochs=EPOCHS_RESNET, lr=LR_RESNET,
                                 on_epoch=lambda h: RESNET_HISTORY.write_text(json.dumps(h, indent=2), encoding="utf-8"))
    resnet_best_epoch = min(resnet_history, key=lambda h: h["val_loss"])["epoch"]
    torch.save({"state_dict": resnet.state_dict(), "model": "resnet50", "n_classes": len(CLASS_LABELS),
                "class_labels": CLASS_LABELS, "seed": SPLIT_SEED, "best_epoch": resnet_best_epoch, "lr": LR_RESNET},
               RESNET_CHECKPOINT)
    print(f"Mejor época: {resnet_best_epoch}. Modelo guardado en {RESNET_CHECKPOINT}")

resnet.to(DEVICE)

### 12.3 Curvas de aprendizaje comparadas

La gráfica de la derecha pone la accuracy de validación frente al **tiempo** de entrenamiento: muestra qué precisión se consigue con el mismo presupuesto de cómputo.

In [ ]:
COLORS = {"CNN": "tab:blue", "ResNet50": "tab:orange"}
histories = {"CNN": history, "ResNet50": resnet_history}

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for name, hist in histories.items():
    color = COLORS[name]
    ep = [h["epoch"] for h in hist]
    hours = np.cumsum([h.get("epoch_time", np.nan) for h in hist]) / 3600
    axes[0].plot(ep, [h["val_loss"] for h in hist], marker="o", color=color, label=name)
    axes[1].plot(ep, [h["val_acc"] for h in hist], marker="o", color=color, label=f"{name} (val)")
    axes[1].plot(ep, [h["train_acc"] for h in hist], linestyle=":", color=color, label=f"{name} (train)")
    axes[2].plot(hours, [h["val_acc"] for h in hist], marker="o", color=color, label=name)
axes[0].set(title="Loss de validación", xlabel="época")
axes[1].set(title="Accuracy", xlabel="época")
axes[2].set(title="Accuracy de validación frente al tiempo", xlabel="horas de entrenamiento")
for ax in axes[:2]:
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

for name, hist in histories.items():
    best = min(hist, key=lambda h: h["val_loss"])
    print(f"{name:<9} mejor época {best['epoch']:>2}: val_acc={best['val_acc']:.4f}  train_acc={best['train_acc']:.4f}  "
          f"brecha train−val={best['train_acc'] - best['val_acc']:+.4f}")

### 12.4 Métricas en test

Se evalúa la ResNet50 sobre el mismo test que la CNN (sección 11). Además de accuracy, top-5 y F1, se muestran la **precision** y el **recall** macro (media de las 50 clases):

- **precision**: de las veces que el modelo predice un radical, cuántas acierta.
- **recall**: de las muestras de un radical, cuántas reconoce.

Los intervalos de confianza del 95 % se calculan por *bootstrap*: se remuestrea el test con reemplazo 1000 veces y se recalcula la métrica.

In [ ]:
def summarize(result: dict) -> dict:
    rep = classification_report(result["y_true"], result["y_pred"], labels=labels, output_dict=True, zero_division=0)
    return {"accuracy": result["acc"], "top-5": result["top5"],
            "precision (macro)": rep["macro avg"]["precision"], "recall (macro)": rep["macro avg"]["recall"],
            "F1 (macro)": rep["macro avg"]["f1-score"], "loss": result["loss"]}


def bootstrap_ci(y_true, y_pred, metric, n: int = 1000, seed: int = 0) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    scores = [metric(y_true[i], y_pred[i]) for i in rng.integers(0, len(y_true), size=(n, len(y_true)))]
    return tuple(np.percentile(scores, [2.5, 97.5]))


resnet_result = evaluate(resnet, test_loader)
results = {"CNN": test_result, "ResNet50": resnet_result}
summary = {name: summarize(r) for name, r in results.items()}
ci_metrics = {"accuracy": lambda t, p: (t == p).mean(),
              "F1 (macro)": lambda t, p: f1_score(t, p, average="macro", zero_division=0)}
ci = {name: {k: bootstrap_ci(r["y_true"], r["y_pred"], f) for k, f in ci_metrics.items()} for name, r in results.items()}

print(f"{'métrica':<18} {'CNN':>24} {'ResNet50':>24} {'Δ (ResNet − CNN)':>17}")
for key in summary["CNN"]:
    cells = []
    for name in results:
        cell = f"{summary[name][key]:.4f}"
        if key in ci[name]:
            cell += " [{:.3f}, {:.3f}]".format(*ci[name][key])
        cells.append(cell)
    print(f"{key:<18} {cells[0]:>24} {cells[1]:>24} {summary['ResNet50'][key] - summary['CNN'][key]:>+17.4f}")

keys = [k for k in summary["CNN"] if k != "loss"]
x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(10, 4.5))
for offset, name in zip((-0.19, 0.19), results):
    values = np.array([summary[name][k] for k in keys])
    err = np.array([[values[j] - ci[name][k][0], ci[name][k][1] - values[j]] if k in ci[name] else [0, 0]
                    for j, k in enumerate(keys)]).T
    bars = ax.bar(x + offset, values, 0.38, yerr=err, capsize=3, label=name, color=COLORS[name])
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
low = min(summary[n][k] for n in results for k in keys)
ax.set_xticks(x, keys)
ax.set(title="Métricas en test", ylim=(max(0, low - 0.1), 1.02))
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 12.5 ¿La diferencia es significativa? Test de McNemar

Los dos modelos se evalúan sobre **las mismas** 7511 imágenes, así que la comparación adecuada es pareada. El test de McNemar ignora las muestras en las que ambos aciertan o ambos fallan y se fija solo en las **discordantes**: si los dos modelos fueran igual de buenos, cada discordancia caería de un lado o del otro con probabilidad 1/2 (test binomial exacto). Un *p-valor* < 0.05 indica que la diferencia no se explica por azar.

In [ ]:
cnn_ok = test_result["y_true"] == test_result["y_pred"]
res_ok = resnet_result["y_true"] == resnet_result["y_pred"]
only_cnn, only_res = int((cnn_ok & ~res_ok).sum()), int((~cnn_ok & res_ok).sum())
p_value = binomtest(only_cnn, only_cnn + only_res, 0.5).pvalue

print(f"{'':<16} {'ResNet acierta':>15} {'ResNet falla':>13}")
print(f"{'CNN acierta':<16} {int((cnn_ok & res_ok).sum()):>15} {only_cnn:>13}")
print(f"{'CNN falla':<16} {only_res:>15} {int((~cnn_ok & ~res_ok).sum()):>13}")
print(f"\nDiscordantes: solo CNN {only_cnn}, solo ResNet50 {only_res}  →  p = {p_value:.2e}")
print("Diferencia significativa (p < 0.05)" if p_value < 0.05 else "Diferencia NO significativa (p ≥ 0.05)")

### 12.6 F1 por radical

Muestra en qué radicales gana cada modelo. Si la hipótesis de la resolución es cierta, la CNN debería ganar en los radicales que se distinguen por trazos finos (日/目/田, 衣/示…).

In [ ]:
f1_cnn = f1_score(test_result["y_true"], test_result["y_pred"], labels=labels, average=None, zero_division=0)
f1_res = f1_score(resnet_result["y_true"], resnet_result["y_pred"], labels=labels, average=None, zero_division=0)
diff = f1_res - f1_cnn
order = np.argsort(diff)

fig, ax = plt.subplots(figsize=(8, 12))
ax.barh([PLOT_NAMES[i] for i in order], diff[order],
        color=[COLORS["ResNet50"] if diff[i] > 0 else COLORS["CNN"] for i in order])
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="F1 por radical: ResNet50 − CNN\n(naranja: mejor ResNet50; azul: mejor CNN)", xlabel="Δ F1")
ax.tick_params(axis="y", labelsize=7)
ax.margins(y=0.01)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ResNet50 mejor en {(diff > 0).sum()} radicales, CNN mejor en {(diff < 0).sum()}, empate en {(diff == 0).sum()}")
for title, idx in (("Más a favor de la CNN:", order[:5]), ("Más a favor de la ResNet50:", order[::-1][:5])):
    print(f"\n{title}")
    for i in idx:
        print(f"  {CLASS_NAMES[i]:>7}  F1 CNN={f1_cnn[i]:.3f}  ResNet50={f1_res[i]:.3f}  Δ={diff[i]:+.3f}")

### 12.7 Dónde discrepan

Muestras de test que un modelo acierta y el otro falla, con la predicción del que falla.

In [ ]:
y_true = test_result["y_true"]
groups = [("acierta la CNN, falla la ResNet50", np.flatnonzero(cnn_ok & ~res_ok), resnet_result["y_pred"], "ResNet"),
          ("acierta la ResNet50, falla la CNN", np.flatnonzero(~cnn_ok & res_ok), test_result["y_pred"], "CNN")]
rng = np.random.default_rng(0)
fig = plt.figure(figsize=(13, 9))
for subfig, (title, idx, wrong_pred, wrong_name) in zip(fig.subfigures(2, 1), groups):
    axes = subfig.subplots(2, 8)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, rng.choice(idx, size=min(16, len(idx)), replace=False)):
        ax.imshow(test_subset[int(i)][0][0], cmap="gray_r")
        ax.set_title(f"real {PLOT_NAMES[y_true[i]]}\n{wrong_name}: {PLOT_NAMES[wrong_pred[i]]}", fontsize=7)
    subfig.suptitle(f"{title} ({len(idx)} muestras)")
plt.show()

### 12.8 Coste computacional

La inferencia se mide sobre un batch de 128 imágenes ya cargado en memoria, para no contar la lectura de ETL9B, que es igual para los dos modelos.

In [ ]:
def time_inference(m: nn.Module, loader: DataLoader, repeats: int = 5) -> float:
    """ms por imagen del forward, sobre un batch ya en memoria."""
    xb = next(iter(loader))[0].to(DEVICE)
    m.eval()
    with torch.no_grad():
        m(xb)  # calentamiento
        start = time.perf_counter()
        for _ in range(repeats):
            m(xb)
    return (time.perf_counter() - start) / (repeats * len(xb)) * 1000


cost = {}
for name, m, hist in (("CNN", model, history), ("ResNet50", resnet, resnet_history)):
    times = [h["epoch_time"] for h in hist if "epoch_time" in h]
    cost[name] = {"parámetros": f"{count_params(m):,}", "tamaño (MB)": f"{size_mb(m):.1f}",
                  "s / época": f"{np.mean(times):.0f}" if times else "—",
                  "ms / imagen": f"{time_inference(m, test_loader):.2f}",
                  "accuracy test": f"{summary[name]['accuracy']:.4f}"}

print(f"{'':<16} {'CNN':>12} {'ResNet50':>12}")
for key in cost["CNN"]:
    print(f"{key:<16} {cost['CNN'][key]:>12} {cost['ResNet50'][key]:>12}")

### 12.9 ¿Por qué?

PORQUE_PLACEHOLDER

## 13. Inferencia

`predict` recibe la imagen de un kanji y devuelve los *k* radicales más probables. Aquí se aplica a muestras aleatorias de test.

In [ ]:
def predict(model: nn.Module, image: torch.Tensor, k: int = 5) -> list[tuple[str, float]]:
    """Top-k radicales para una imagen (1, 63, 64)."""
    model.eval()
    with torch.no_grad():
        probs = model(image.unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu()
    top = probs.topk(k)
    return [(CLASS_NAMES[c], p) for p, c in zip(top.values.tolist(), top.indices.tolist())]


rng = np.random.default_rng(7)
fig, axes = plt.subplots(2, 6, figsize=(16, 6), gridspec_kw={"height_ratios": [1.2, 1]})
for col, i in enumerate(rng.choice(len(test_subset), size=6, replace=False)):
    image, true_class = test_subset[int(i)]
    top = predict(model, image)
    ok = top[0][0] == CLASS_NAMES[true_class]
    axes[0, col].imshow(image[0], cmap="gray_r")
    axes[0, col].set_title(f"real {PLOT_NAMES[true_class]}", color="tab:green" if ok else "tab:red")
    axes[0, col].axis("off")
    names_k = [n if HAS_CJK_FONT else n.split()[0] for n, _ in top][::-1]
    axes[1, col].barh(names_k, [p for _, p in top][::-1], color="tab:blue")
    axes[1, col].set_xlim(0, 1)
    axes[1, col].set_xlabel("probabilidad")
fig.suptitle("Predicciones top-5 sobre muestras de test")
plt.tight_layout()
plt.show()

## 14. Conclusiones

Puntos para interpretar los resultados de las secciones 10–12:

- **Generalización real.** Como test no comparte escritores con train, su accuracy mide cuánto acierta la red con la letra de personas nuevas. Una diferencia pequeña entre val y test indica que la elección de la mejor época no sobreajustó a val.
- **Top-5 frente a top-1.** Si el top-5 es mucho mayor que la accuracy, la red suele dudar entre unos pocos radicales parecidos, no se equivoca por completo.
- **Radicales difíciles.** En una prueba corta (2 épocas), los de menor accuracy fueron componentes simples y muy frecuentes (日, 木, 田, 禾, 口). Aparecen en muchos kanji, a veces como parte de otros componentes, y la regla de etiquetado del primer nivel IDS puede asignar a un kanji un radical que visualmente no domina. La matriz de confusión y la lista de confusiones más frecuentes muestran qué pares se mezclan.
- **Limitaciones y mejoras posibles.** El etiquetado excluye los kanji con varios radicales en el primer nivel; un enfoque multietiqueta los aprovecharía. Más épocas, una red más profunda (p. ej. ResNet) o más augmentación podrían mejorar los radicales difíciles.